# Évaluation gain/risque indépendante (holdout vierge, 2 GPU)

Protocole **distinct** de celui des étudiants :

- **Calibration de θ** sur leur périmètre (split held-out *par alerte* du **train**), puis θ **figé**.
- **Test final** sur un **holdout jamais vu** (l'éval officielle, non distribuée aux étudiants).
- **Règle d'arrêt « first-passage »** : par alerte, on valide la 1ʳᵉ prédiction (ordre chronologique) de confiance ≥ θ, puis on s'arrête.
- **Gain jamais négatif** : `t_eff = min(t_pred, baseline_end)` ⇒ `gain = max(0, baseline_end − t_pred)`.
- **Deux règles** : `CG` (t_first/t_last/risque sur CG seuls) et `CG+IC` (tout type compte).
- **Risque** `missed / total` dans la zone `dist < d`, dénominateur global, pour **d = 5 km** (contrainte θ < 2 %) et **d = 20 km** (informatif).
- **Baseline 30 min** : ancre, gain 0 et risque 0 par construction.

> ⚠️ Inférence à exécuter sur l'**env GPU** (2× T4). En local CPU, n'exécuter que le chargement et la logique d'évaluation.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

import eval_inference as ei
import gain_risk_eval as gr
import train_models as tm

# ── Constantes (à adapter) ──────────────────────────────────────────────
# Les étudiants n'avaient QUE le train → split fit/calib PARTAGÉ avec l'entraînement.
TRAIN_CSV = "/home/vm2lucas/Bureau/Hackathon/hackathon2026/data/segment_alerts_all_airports_train.csv"
# Holdout vierge jamais vu par les étudiants (schéma jury). NOM À RENSEIGNER :
HOLDOUT_CSV = "A_RENSEIGNER.csv"
# Par défaut, le held-out conservé est l'éval officielle non distribuée :
# HOLDOUT_CSV = "/home/vm2lucas/Bureau/Hackathon/hackathon2026/data/segment_alerts_all_airports_eval.csv"

MODELS_DIR = REPO / "models"
SEED = tm.SEED                       # split reproductible, IDENTIQUE à l'entraînement
CALIB_FRAC = tm.CALIB_FRAC           # fraction du train réservée à la calibration de θ
TRAIN_MODELS = False                 # True → (ré)entraîne tous les modèles sur le fit (GPU)
ACCEPTABLE_RISK = gr.ACCEPTABLE_RISK  # 2 %
RULES = list(gr.RULES)               # ['CG', 'CG+IC']
ZONES_KM = gr.ZONES_KM               # (5.0, 20.0)
N_MC = ei.N_MC                       # passes MC Dropout

## 1. Données : split fit/calib partagé + holdout vierge

`make_split` sépare le train **par alerte** en `fit` (entraînement) et `calib` (calibration de θ). Le **même** appel sert à l'entraînement → les modèles ne voient jamais les alertes de `calib`, donc θ est calibré **sans fuite**.

In [ ]:
# Split par alerte partagé avec l'entraînement (même seed/frac).
fit_alerts, calib_alerts = tm.make_split(TRAIN_CSV, seed=SEED, calib_frac=CALIB_FRAC)
holdout_alerts = ei.load_eval_alerts(HOLDOUT_CSV)


def _n_alerts(df):
    return df.groupby(["airport", "airport_alert_id"]).ngroups


print(f"Fit (entraînement)     : {_n_alerts(fit_alerts)} alertes / {len(fit_alerts)} éclairs")
print(f"Calibration θ (vierge) : {_n_alerts(calib_alerts)} alertes / {len(calib_alerts)} éclairs")
print(f"Holdout (test final)   : {_n_alerts(holdout_alerts)} alertes / {len(holdout_alerts)} éclairs")

## 1bis. (Optionnel) Entraînement contrôlé sur le fit

`TRAIN_MODELS=True` (ré)entraîne **tous** les modèles du registre sur la portion `fit`
uniquement, sur les 2 GPU. La portion `calib` reste vierge pour θ.

In [ ]:
# (Ré)entraînement contrôlé : TOUS les modèles du registre, sur le fit UNIQUEMENT, 2 GPU.
# À exécuter sur l'env GPU. Sinon laisser TRAIN_MODELS=False et déposer des checkpoints
# compatibles dans MODELS_DIR (mêmes noms que eval_inference.SEQ_REGISTRY).
if TRAIN_MODELS:
    devices = ei.detect_devices()
    tm.train_sequential(fit_alerts, MODELS_DIR, devices, verbose=True)
    tm.train_tabular(fit_alerts, MODELS_DIR, devices[0], seed=SEED)
    print("Entraînement terminé.")
else:
    print("TRAIN_MODELS=False — checkpoints supposés déjà présents dans", MODELS_DIR)

## 2. Inférence batchée multi-GPU (calibration + holdout)

Émet une prédiction à **chaque éclair** de **chaque alerte** (rien n'est écarté).
Distribue les sessions de chaque modèle sur tous les `cuda:i` disponibles.

In [ ]:
devices = ei.detect_devices()
print("Devices :", [str(d) for d in devices])

preds_calib = ei.run_all_inference(calib_alerts, models_dir=MODELS_DIR, devices=devices, n_mc=N_MC)
preds_holdout = ei.run_all_inference(holdout_alerts, models_dir=MODELS_DIR, devices=devices, n_mc=N_MC)

if not preds_calib:
    print("\n⚠️ Aucun checkpoint dans", MODELS_DIR, "— entraîner/déposer les .pt d'abord.")
else:
    for name, df in preds_holdout.items():
        print(f"  {name:26s} | {len(df):7d} prédictions (holdout)")

## 3. Calibration de θ (sur calib) puis application figée (holdout)

In [ ]:
sweeps = {}  # (modèle, règle) -> sweep_df, pour les figures
rows = []


def _evaluate_model(name, preds_c, preds_h):
    for rule in RULES:
        cal = gr.calibrate_theta(preds_c, calib_alerts, rule, ACCEPTABLE_RISK)
        ap = gr.apply_holdout(preds_h, holdout_alerts, cal["theta_star"], rule, ZONES_KM)
        sweeps[(name, rule)] = cal["sweep_df"]
        rows.append({
            "modele": name, "regle": rule,
            "theta*": cal["theta_star"], "faisable": cal["feasible"],
            "gain_calib_h": cal["gain_h"], "risk5_calib": cal["risk_5km"],
            "gain_holdout_h": ap["gain_h"],
            "risk5_holdout": ap["risk_5km"], "risk20_holdout": ap["risk_20km"],
            "n_couvertes": ap["n_covered"], "n_alertes": ap["n_alerts"],
        })


for name in preds_calib:
    _evaluate_model(name, preds_calib[name], preds_holdout[name])

# Baseline 30 min (ancre, rule-aware) : gain 0 / risque 0 par construction.
for rule in RULES:
    base_c = ei.baseline_predictions(calib_alerts, rule)
    base_h = ei.baseline_predictions(holdout_alerts, rule)
    _evaluate_model("Baseline 30 min", base_c, base_h)

results = pd.DataFrame(rows)
results

## 4. Tableau récapitulatif (holdout)

Gain et risque **sur le holdout** au θ* calibré, par modèle et par règle. La colonne
`faisable` indique si la contrainte risque 5 km < 2 % a pu être tenue **en calibration**.

In [ ]:
view = results.pivot_table(
    index="modele",
    columns="regle",
    values=["theta*", "gain_holdout_h", "risk5_holdout", "risk20_holdout"],
    aggfunc="first",
).sort_values(("gain_holdout_h", "CG+IC"), ascending=False)


def _flag_risk(v):
    return "color: red" if v >= ACCEPTABLE_RISK else "color: green"


view.style.format("{:.3f}").map(_flag_risk, subset=["risk5_holdout", "risk20_holdout"])

## 5. Figures

In [ ]:
# (a) Compromis gain ↔ risque 5 km sur le holdout, une figure par règle.
fig, axes = plt.subplots(1, len(RULES), figsize=(6 * len(RULES), 5), squeeze=False)
for ax, rule in zip(axes[0], RULES):
    sub = results[results["regle"] == rule]
    ax.scatter(sub["risk5_holdout"], sub["gain_holdout_h"], s=60)
    for _, r in sub.iterrows():
        ax.annotate(r["modele"], (r["risk5_holdout"], r["gain_holdout_h"]),
                    fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax.axvline(ACCEPTABLE_RISK, color="red", ls="--", label="risque 2 %")
    ax.set(xlabel="risque 5 km (holdout)", ylabel="gain (h, holdout)", title=f"Règle {rule}")
    ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# (b) Courbe de sweep θ (calibration) pour un modèle donné : gain vs risque 5 km.
MODEL = next((n for n in preds_calib), None)  # 1er modèle ; à changer au besoin
if MODEL is not None:
    fig, axes = plt.subplots(1, len(RULES), figsize=(6 * len(RULES), 4), squeeze=False)
    for ax, rule in zip(axes[0], RULES):
        s = sweeps[(MODEL, rule)]
        ax2 = ax.twinx()
        ax.plot(s["theta"], s["gain_h"], "o-", color="tab:blue", label="gain (h)")
        ax2.plot(s["theta"], s["risk"], "s--", color="tab:red", label="risque 5 km")
        ax2.axhline(ACCEPTABLE_RISK, color="red", ls=":", alpha=0.6)
        ax.set(xlabel="θ", ylabel="gain (h)", title=f"{MODEL} — {rule}")
        ax2.set_ylabel("risque 5 km")
    fig.tight_layout()
    plt.show()